In [1]:
!pip -q install gensim scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 37.4 MB/s eta 0:00:00


In [2]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
import requests

url = "https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/gu.txt"

response = requests.get(url, stream=True)
response.raise_for_status()

print("Dataset connection successful!")

Dataset connection successful!


In [4]:
sentences = []

for line in response.iter_lines():
    if line:
        sentence = line.decode("utf-8").strip()

        if sentence:
            sentences.append(sentence)

    if len(sentences) >= 1_000_000:
        break

print("Total sentences:", len(sentences))

print("\nFirst 5 sentences:\n")

for s in sentences[:5]:
    print(s)

Total sentences: 1000000

First 5 sentences:

આ વીડિયો જુઓ: ઊંઝા માર્કેટયાર્ડ આજથી 25 જુલાઈ સુધી બંધ
મિથેનોલ આવ્યો ક્યાંથી?
આખરે ત્રણ રાજ્યોમાં મળેલ હાર પર કોંગ્રેસ અધ્યક્ષ રાહુલ ગાંધી દ્વારા પ્રથમ પ્રતિક્રિયા આપવામાં આવી છે. તેમણે કહ્યું કે, ત્રિપુરા, નાગાલેન્ડ અને મેઘાલયમાં લોકોના જનાદેશનો સ્વાગત કરીએ છે અને આ ક્ષેત્રના લોકોનો વિશ્વાસ ફરીથી જીતીવા માટે પ્રતિબદ્ધ છીએ.
આ આંકડો માટે, અને વજન ઘટાડવા માટે પ્રકાશનનો દિવસ વિતાવવો ઉપયોગી છે, ઉદાહરણ તરીકે, અઠવાડિયામાં એક વખત. તમારા માટે એક વિકલ્પ પસંદ કરો જે અગવડતાને કારણે નહીં કરે. સૌથી વધુ લોકપ્રિય કીફિર પર અનલોડ છે.
આ ઠેકાઓ પરથી લીમડી તેમજ ઝાલોદના બૂટલેગરો વિદેશી દારૃનો મોટાપાયે જથ્થો ખરીદી મહિસાગર, હાલોલ, ગોધરા, આણંદ, નડિયાદ અને વડોદરા શહેર-જિલ્લામાં ઠાલવે છે. આબુથી આવતો દારૃનો જથ્થો સાંચોર તેમજ પાલનપુર થઈ મહેસાણામાં રીટા અને વિરસિંહને ત્યાં ઉતારવામાં આવે છે, જ્યાંથી અમદાવાદ તેમજ ઉત્તર ગુજરાતમાં દારૃ સપ્લાઈ થાય છે. રાજસ્થાનના બિચ્છુવાડાથી શામળાજી બોર્ડર થઈ હિંમતનગર, ગાંધીનગર અને અમદાવાદમાં બૂટલેગર સુનિલ, વિનોદ, દિલીપ અને રબારી દ્વારા મોટા

In [5]:
random.seed(42)

random.shuffle(sentences)

train_sentences = sentences[:800000]
dev_sentences = sentences[800000:900000]
test_sentences = sentences[900000:1000000]

print("Training sentences:", len(train_sentences))
print("Validation sentences:", len(dev_sentences))
print("Test sentences:", len(test_sentences))

Training sentences: 800000
Validation sentences: 100000
Test sentences: 100000


In [6]:
def tokenize(sentence):
    return re.findall(r'\w+|[.,!?;:]', sentence.lower())

train_tokens = [tokenize(s) for s in train_sentences]

print("Tokenization completed!")
print("\nExample:")
print(train_tokens[0])

Tokenization completed!

Example:
['સ', 'મ', 'ર', 'ટફ', 'ન', 'લ', 'વ', 'છ', 'પર', 'ત', 'તમન', 'આ', 'નથ', 'ખબર', 'ત', 'ક', 'ઇ', 'જ', 'નથ', 'ખબર']


In [7]:
print("Number of training sentences:", len(train_tokens))
print("Example sentence:")
print(train_sentences[0])

print("\nNumber of words in first sentence:", len(train_tokens[0]))

Number of training sentences: 800000
Example sentence:
સ્માર્ટફોન લેવો છે પરંતુ તમને આ નથી ખબર તો કંઇ જ નથી ખબર

Number of words in first sentence: 20


In [8]:
from gensim.models import Word2Vec

print("Training Word2Vec...")

word2vec_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=3
)

print("Word2Vec training completed!")
print("Vocabulary size:", len(word2vec_model.wv))

Training Word2Vec...
Word2Vec training completed!
Vocabulary size: 25358


In [9]:
word = "ગુજરાત"

if word in word2vec_model.wv:
    print("Vector for", word)
    print(word2vec_model.wv[word][:10])

    print("\nSimilar words:")
    print(word2vec_model.wv.most_similar(word, topn=10))
else:
    print("Word not found in vocabulary.")

Word not found in vocabulary.


In [10]:
word2vec_model.save("/content/word2vec_gujarati.model")

print("Word2Vec model saved successfully!")

Word2Vec model saved successfully!


In [11]:
from gensim.models import FastText

print("Training FastText...")

fasttext_model = FastText(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=3
)

print("FastText training completed!")
print("Vocabulary size:", len(fasttext_model.wv))

Training FastText...
FastText training completed!
Vocabulary size: 25358


In [12]:
word = "ગુજરાત"

print("Vector for:", word)
print(fasttext_model.wv[word][:10])

print("\nSimilar words:")
print(fasttext_model.wv.most_similar(word, topn=10))

Vector for: ગુજરાત
[ 0.00074118 -0.00062506 -0.00022135  0.00318854  0.01175855 -0.00851154
 -0.00150701  0.00741882  0.00084076  0.00152063]

Similar words:
[('મઢડ', 0.6308173537254333), ('ફણ', 0.6305736899375916), ('ઢડ', 0.6144208908081055), ('લપર', 0.6027445197105408), ('તલપર', 0.597949206829071), ('બગસર', 0.5912814140319824), ('ઠલનગર', 0.5893739461898804), ('૩ર૮', 0.5874819755554199), ('ગઢડ', 0.5856762528419495), ('મલપર', 0.5855916738510132)]


In [13]:
fasttext_model.save("/content/fasttext_gujarati.model")

print("FastText model saved successfully!")

FastText model saved successfully!


In [14]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

print("Preparing documents for Doc2Vec...")

tagged_documents = [
    TaggedDocument(words=tokens, tags=[i])
    for i, tokens in enumerate(train_tokens)
]

print("Documents prepared:", len(tagged_documents))

Preparing documents for Doc2Vec...
Documents prepared: 800000


In [15]:
print("Training Doc2Vec...")

doc2vec_model = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=5,
    dm=1
)

doc2vec_model.build_vocab(tagged_documents)

print("Vocabulary built!")
print("Vocabulary size:", len(doc2vec_model.wv))

Training Doc2Vec...
Vocabulary built!
Vocabulary size: 25358


In [16]:
doc2vec_model.train(
    tagged_documents,
    total_examples=len(tagged_documents),
    epochs=doc2vec_model.epochs
)

print("Doc2Vec training completed!")

Doc2Vec training completed!


In [17]:
test_doc = train_tokens[0]

vector = doc2vec_model.infer_vector(test_doc)

print("Doc2Vec vector shape:", vector.shape)
print("First 10 values:")
print(vector[:10])

Doc2Vec vector shape: (100,)
First 10 values:
[-0.01560602 -0.11853043  0.01811939 -0.10402419  0.01135241  0.22264828
  0.05634025 -0.10668812  0.0072391   0.10109015]


In [18]:
doc2vec_model.save("/content/doc2vec_gujarati.model")

print("Doc2Vec model saved successfully!")

Doc2Vec model saved successfully!


In [19]:
from sklearn.cluster import KMeans
import numpy as np

# Get words and their Word2Vec vectors
words = list(word2vec_model.wv.index_to_key)

word_vectors = np.array([
    word2vec_model.wv[word]
    for word in words
])

print("Number of words:", len(words))
print("Vector shape:", word_vectors.shape)

Number of words: 25358
Vector shape: (25358, 100)


In [20]:
print("Running K-Means clustering...")

K = 10

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(word_vectors)

print("K-Means completed!")
print("Number of clusters:", K)

Running K-Means clustering...
K-Means completed!
Number of clusters: 10


In [21]:
print("20 closest words for each cluster:\n")

centroids = kmeans.cluster_centers_

for cluster_id in range(K):

    # Words belonging to this cluster
    distances = np.linalg.norm(
        word_vectors - centroids[cluster_id],
        axis=1
    )

    # Get 20 closest words to centroid
    closest_indices = np.argsort(distances)[:20]

    closest_words = [words[i] for i in closest_indices]

    print(f"Cluster {cluster_id + 1}:")
    print(", ".join(closest_words))
    print()

20 closest words for each cluster:

Cluster 1:
activists, foot, poster, created, once, come, increased, thumb, another, resignation, parents, cars, corporator, gets, until, wins, changed, beat, contest, joy

Cluster 2:
૫૯, ૭૭, ૫૭, ૫૪, ૭૯, ૮૭, ૫૮, ૬૪, ૮૮, ૪૯, ૭૪, ૬૨, ૮૬, ૯૨, ૮૪, ૫૬, ૭૨, ૫૧, ૪૭, ૬૩

Cluster 3:
હલનચલન, બરછટ, ગરદનન, ઉઝરડ, ચમકવ, આવરણ, પડધ, ઘનત, rhinestones, ઊગવ, નસક, મલમ, એકસરખ, sleeves, કફન, ખરબચડ, મણક, સહનશક, ચપળ, દણ

Cluster 4:
મગનભ, કરશનભ, અજયભ, છગનભ, રમભ, અરજણભ, મલભ, કજભ, રમણભ, વનભ, રવજ, રણભ, રધનભ, ગરભ, તનભ, બભ, શનભ, મદભ, દશરથભ, પટભ

Cluster 5:
04, 06, 07, 94, 78, 93, 09, 91, 88, 101, 79, 110, 83, 77, 92, 61, 125, 105, 86, 69

Cluster 6:
ઢસડ, ચઢત, હવસખ, મગરન, ઈકસવ, આખલ, ગટરમ, ચઢવ, ભડથ, કઅપમ, ઘસડ, ઉચક, પકડત, કમરમ, અજગર, ઢન, ઝપટ, ચડય, લથપથ, દવમ

Cluster 7:
તનથ, ડકપમ, ડકપન, તનમ, કઆઉટ, ણવવ, ગળત, ગખ, કઆઉટન, ફળસ, ડકપ, ચહર, તનશ, ષભર, ષપલટ, ણનમ, સનફ, ણનન, નસર, પદક

Cluster 8:
સદશ, રમણલ, વલભ, દશરથ, વરનગર, સરધ, જયગ, નટવરલ, રલબ, કરમસદ, શનગર, છગનલ, ઠકકર, મગનલ, થધ, સતવ, ઠકર, ગરસ, સ

In [22]:
import pandas as pd

cluster_results = []

for cluster_id in range(K):
    distances = np.linalg.norm(
        word_vectors - centroids[cluster_id],
        axis=1
    )

    closest_indices = np.argsort(distances)[:20]
    closest_words = [words[i] for i in closest_indices]

    cluster_results.append({
        "Cluster": cluster_id + 1,
        "Words": ", ".join(closest_words)
    })

cluster_df = pd.DataFrame(cluster_results)

display(cluster_df)

cluster_df.to_csv(
    "/content/kmeans_clusters.csv",
    index=False
)

print("Cluster results saved!")

,Cluster,Words
0,1,"activists, foot, poster, created, once, come, ..."
1,2,"૫૯, ૭૭, ૫૭, ૫૪, ૭૯, ૮૭, ૫૮, ૬૪, ૮૮, ૪૯, ૭૪, ૬૨..."
2,3,"હલનચલન, બરછટ, ગરદનન, ઉઝરડ, ચમકવ, આવરણ, પડધ, ઘન..."
3,4,"મગનભ, કરશનભ, અજયભ, છગનભ, રમભ, અરજણભ, મલભ, કજભ,..."
4,5,"04, 06, 07, 94, 78, 93, 09, 91, 88, 101, 79, 1..."
5,6,"ઢસડ, ચઢત, હવસખ, મગરન, ઈકસવ, આખલ, ગટરમ, ચઢવ, ભડ..."
6,7,"તનથ, ડકપમ, ડકપન, તનમ, કઆઉટ, ણવવ, ગળત, ગખ, કઆઉટ..."
7,8,"સદશ, રમણલ, વલભ, દશરથ, વરનગર, સરધ, જયગ, નટવરલ, ..."
8,9,"આળસ, સહજત, ઘરડ, વનભર, મનથ, આચરણ, અફસ, નફરત, વશ..."
9,10,"ઑટ, એડલ, ટટ, શયલ, આઇક, ડટ, ઇનફ, led, ડડ, ઑથ, ઇ..."


Cluster results saved!


In [23]:
# Free memory that is no longer needed
del tagged_documents

import gc
gc.collect()

print("Unused memory cleared.")

Unused memory cleared.


In [24]:
def word2vec_sentence_vector(tokens, model):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [25]:
def fasttext_sentence_vector(tokens, model):
    vectors = [
        model.wv[word]
        for word in tokens
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [26]:
print("Tokenizing validation and test sentences...")

dev_tokens = [tokenize(s) for s in dev_sentences]
test_tokens = [tokenize(s) for s in test_sentences]

print("Validation sentences:", len(dev_tokens))
print("Test sentences:", len(test_tokens))

Tokenizing validation and test sentences...
Validation sentences: 100000
Test sentences: 100000


In [27]:
print("Creating Word2Vec sentence vectors...")

dev_w2v = np.array([
    word2vec_sentence_vector(tokens, word2vec_model)
    for tokens in dev_tokens
], dtype=np.float32)

test_w2v = np.array([
    word2vec_sentence_vector(tokens, word2vec_model)
    for tokens in test_tokens
], dtype=np.float32)

print("Validation shape:", dev_w2v.shape)
print("Test shape:", test_w2v.shape)

Creating Word2Vec sentence vectors...
Validation shape: (100000, 100)
Test shape: (100000, 100)


In [28]:
print("Creating FastText sentence vectors...")

dev_ft = np.array([
    fasttext_sentence_vector(tokens, fasttext_model)
    for tokens in dev_tokens
], dtype=np.float32)

test_ft = np.array([
    fasttext_sentence_vector(tokens, fasttext_model)
    for tokens in test_tokens
], dtype=np.float32)

print("Validation shape:", dev_ft.shape)
print("Test shape:", test_ft.shape)

Creating FastText sentence vectors...
Validation shape: (100000, 100)
Test shape: (100000, 100)


In [29]:
print("Creating Doc2Vec sentence vectors...")

dev_d2v = np.array([
    doc2vec_model.infer_vector(tokens)
    for tokens in dev_tokens
], dtype=np.float32)

test_d2v = np.array([
    doc2vec_model.infer_vector(tokens)
    for tokens in test_tokens
], dtype=np.float32)

print("Validation shape:", dev_d2v.shape)
print("Test shape:", test_d2v.shape)

Creating Doc2Vec sentence vectors...
Validation shape: (100000, 100)
Test shape: (100000, 100)


In [30]:
from sklearn.neighbors import NearestNeighbors

print("Nearest-neighbor search ready.")

Nearest-neighbor search ready.


In [31]:
print("Finding closest sentences using Word2Vec...")

nn_w2v = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute"
)

nn_w2v.fit(test_w2v)

distances_w2v, indices_w2v = nn_w2v.kneighbors(dev_w2v)

print("Word2Vec sentence matching completed!")

Finding closest sentences using Word2Vec...
Word2Vec sentence matching completed!


In [32]:
for i in range(10):
    closest_index = indices_w2v[i][0]
    similarity = 1 - distances_w2v[i][0]

    print(f"\nValidation sentence {i+1}:")
    print(" ", dev_sentences[i])

    print("Closest test sentence:")
    print(" ", test_sentences[closest_index])

    print("Cosine similarity:", round(similarity, 4))


Validation sentence 1:
  પ્રાપ્ત માહિતી મુજબ,  મોડાસા-શામળાજી રોડ પર આવેલા ખોડંબા ગામ નજીક ખેતરમાંથી બળદગાડું લઈ પરત ઘરે ફરતા પ્રભાભાઇ શામળભાઈ પટેલ ઘર નજીક પહોંચ્યા હતા શામળાજી તરફથી આવતી એમ્બ્યુલન્સે બળદગાડાને ધડાકાભેર અડફેટે લેતા બળદગાડું હંકારતા પ્રભાભાઇ બળદગાડાંમાંથી નીચે પટકાતા શરીરે ગંભીર ઈજાઓ પહોંચતા ઘટનાસ્થળે કમકમાટી ભર્યું મોત નિપજતા ભારે ચકચાર મચી હતી બળદગાડા સાથે રહેલા બળદો પણ ઇજગ્રસ્ત થયા હતા ઘર નજીક અકસ્માત થતા ખેડૂતના પરિવારજનો અને ગ્રામજનોના ઘટનાસ્થળે દોડી આવ્યા હતા.
Closest test sentence:
  • ચાવંડ પાસે કાર પલટી ખાઈ જતાં દંપતીનું મૃત્યુઃ અમરેલીના જીવરાજપાર્કમાં રહેતા અને બીએસએનએલમાં સબ ડિવિ. એન્જિયર તરીકે ફરજ બચાવતા પ્રવીણભાઈ હરજીભાઈ કવા (ઉ. ૫૮), તેમનાં પત્ની રમાબહેન (ઉ. ૫૫), પુત્ર નિરવ અને પુત્રવધૂ ભૂમિકાબહેન કાર લઈને ગાંધીનગર રહેતા સંબંધીને મળીને પરત અમરેલી આવવા નીકળ્યા હતા. દરમિયાન ચાવંડ-ઢસા હાઇવે પર કાર પલટી ખાઈ જતાં પ્રવીણભાઈ અને રમાબહેનનું ઘટના સ્થળે જ મૃત્યુ નીપજ્યું હતું તથા નિરવ અને ભૂમિકાબહેનને નજીકની હોસ્પિટલે ખસેડાયા હતા. આ બનાવ અંગેની જાણ થતા મૃતક પ્રવીણભા

In [33]:
print("Finding closest sentences using FastText...")

nn_ft = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute"
)

nn_ft.fit(test_ft)

distances_ft, indices_ft = nn_ft.kneighbors(dev_ft)

print("FastText sentence matching completed!")

Finding closest sentences using FastText...
FastText sentence matching completed!


In [34]:
for i in range(10):
    closest_index = indices_ft[i][0]
    similarity = 1 - distances_ft[i][0]

    print(f"\nValidation sentence {i+1}:")
    print(" ", dev_sentences[i])

    print("Closest test sentence:")
    print(" ", test_sentences[closest_index])

    print("Cosine similarity:", round(similarity, 4))


Validation sentence 1:
  પ્રાપ્ત માહિતી મુજબ,  મોડાસા-શામળાજી રોડ પર આવેલા ખોડંબા ગામ નજીક ખેતરમાંથી બળદગાડું લઈ પરત ઘરે ફરતા પ્રભાભાઇ શામળભાઈ પટેલ ઘર નજીક પહોંચ્યા હતા શામળાજી તરફથી આવતી એમ્બ્યુલન્સે બળદગાડાને ધડાકાભેર અડફેટે લેતા બળદગાડું હંકારતા પ્રભાભાઇ બળદગાડાંમાંથી નીચે પટકાતા શરીરે ગંભીર ઈજાઓ પહોંચતા ઘટનાસ્થળે કમકમાટી ભર્યું મોત નિપજતા ભારે ચકચાર મચી હતી બળદગાડા સાથે રહેલા બળદો પણ ઇજગ્રસ્ત થયા હતા ઘર નજીક અકસ્માત થતા ખેડૂતના પરિવારજનો અને ગ્રામજનોના ઘટનાસ્થળે દોડી આવ્યા હતા.
Closest test sentence:
  પાનવાડી વિસ્તરમાં આવેલ સરદાર બાગના બગીચાના ગેટ નજીક ગોપાલ ડાભી અને તેના મિત્રો બેઠા હતા તે સમયે અચાનક ત્રણથી ચાર ઈસમો ઘાતકી હથિયારો સાથે ત્યાં આવી ગોપાલ ડાભી પર ખૂની હુમલો કરી છરી તલવાર જેવા હથિયારોના ઘા મારી ગંભીર ઈજાઓ પહોંચાડી હત્યા કરી ફરાર થઈ ગયા હતા, આ હત્યાના બનાવ બાદ બીજે દિવસે પાનવાડી વિસ્તારમાં તોફાન ના છમકલા પણ થયા હતા અને કેટલીક દુકાનોમાં તોડફોડ કરવામાં આવી હતી. તે ઉપરાંત એક કેબીનમાં આગ પણ લગડવામાં આવી હતી, જો કે પોલીસે જે તે સમયે સ્થિતિ પર કાબુ મેળવી લીધો હતો.
Cosine sim

In [36]:
print("Finding closest sentences using Doc2Vec...")

nn_d2v = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute"
)

nn_d2v.fit(test_d2v)

distances_d2v, indices_d2v = nn_d2v.kneighbors(dev_d2v)

print("Doc2Vec sentence matching completed!")

Finding closest sentences using Doc2Vec...
Doc2Vec sentence matching completed!


In [37]:
for i in range(10):
    closest_index = indices_d2v[i][0]
    similarity = 1 - distances_d2v[i][0]

    print(f"\nValidation sentence {i+1}:")
    print(" ", dev_sentences[i])

    print("Closest test sentence:")
    print(" ", test_sentences[closest_index])

    print("Cosine similarity:", round(similarity, 4))


Validation sentence 1:
  પ્રાપ્ત માહિતી મુજબ,  મોડાસા-શામળાજી રોડ પર આવેલા ખોડંબા ગામ નજીક ખેતરમાંથી બળદગાડું લઈ પરત ઘરે ફરતા પ્રભાભાઇ શામળભાઈ પટેલ ઘર નજીક પહોંચ્યા હતા શામળાજી તરફથી આવતી એમ્બ્યુલન્સે બળદગાડાને ધડાકાભેર અડફેટે લેતા બળદગાડું હંકારતા પ્રભાભાઇ બળદગાડાંમાંથી નીચે પટકાતા શરીરે ગંભીર ઈજાઓ પહોંચતા ઘટનાસ્થળે કમકમાટી ભર્યું મોત નિપજતા ભારે ચકચાર મચી હતી બળદગાડા સાથે રહેલા બળદો પણ ઇજગ્રસ્ત થયા હતા ઘર નજીક અકસ્માત થતા ખેડૂતના પરિવારજનો અને ગ્રામજનોના ઘટનાસ્થળે દોડી આવ્યા હતા.
Closest test sentence:
  હિંમતનગર બાયપાસ રોડ નજીક ઓવરલોડ ખનીજ પસાર  થતા ચાર ટ્રકોને ખાણખનીજ વિભાગની ટીમે ઝડપી પાડી
Cosine similarity: 0.6698

Validation sentence 2:
  જ્યાં સંબંધ હોય છે ત્યાં મુક્તિનો અહેસાસ હોય છે. જ્યાં બંધન હોય છે ત્યાં બંધિયારપણું ને ગૂંગળામણ હોય છે. સંબંધો માનવી પોતે આપબળે વ્યક્તિગત આવડતથી કેળવે છે, જ્યારે બંધનો માનવી ઉપર સામાજિક રીતિ-રિવાજ અને પરંપરાને કારણે થોપવામાં આવે છે. સંબંધમાં કાર્યશીલ રહેવા માટે માનવીની અંદરથી ઉમળકો ઊઠે છે. જ્યારે બંધનમાં માનવીએ બાહ્ય દબાણથી વશ બનીને કામ કરવું

In [38]:
results = []

for i in range(len(dev_sentences)):
    # Word2Vec
    idx_w2v = indices_w2v[i][0]
    sim_w2v = 1 - distances_w2v[i][0]

    # FastText
    idx_ft = indices_ft[i][0]
    sim_ft = 1 - distances_ft[i][0]

    # Doc2Vec
    idx_d2v = indices_d2v[i][0]
    sim_d2v = 1 - distances_d2v[i][0]

    results.append({
        "Validation Sentence": dev_sentences[i],
        "W2V Closest Test Sentence": test_sentences[idx_w2v],
        "W2V Similarity": sim_w2v,
        "FastText Closest Test Sentence": test_sentences[idx_ft],
        "FastText Similarity": sim_ft,
        "Doc2Vec Closest Test Sentence": test_sentences[idx_d2v],
        "Doc2Vec Similarity": sim_d2v
    })

results_df = pd.DataFrame(results)

print("Results created!")
display(results_df.head(10))

Results created!


,Validation Sentence,W2V Closest Test Sentence,W2V Similarity,FastText Closest Test Sentence,FastText Similarity,Doc2Vec Closest Test Sentence,Doc2Vec Similarity
0,"પ્રાપ્ત માહિતી મુજબ, મોડાસા-શામળાજી રોડ પર આવ...",• ચાવંડ પાસે કાર પલટી ખાઈ જતાં દંપતીનું મૃત્યુ...,0.971690,પાનવાડી વિસ્તરમાં આવેલ સરદાર બાગના બગીચાના ગેટ...,0.936294,હિંમતનગર બાયપાસ રોડ નજીક ઓવરલોડ ખનીજ પસાર થતા...,0.669756
1,જ્યાં સંબંધ હોય છે ત્યાં મુક્તિનો અહેસાસ હોય છ...,"મેહતા સાહેબ સવારે ઉઠે છે, ઉઠીને ચોક્કસ સમયે પ્...",0.990059,"મેહતા સાહેબ સવારે ઉઠે છે, ઉઠીને ચોક્કસ સમયે પ્...",0.972246,પ્રધાનમંત્રી મોદીએ BRICS સંમેલનના સંબોધનમાં એવ...,0.598397
2,મ્યુચ્યુઅલ ફંડ માટે નીતિને ઉલ્ટાવવામાં નહીં આવ...,'હ્યૂમન રાઇટ્સ વૉચ'ના આધારે આ ફંડની મોટા ભાગની...,0.950602,માર્બલ માટે 2014માં દાદા સાહેબ ફાલ્કે ફિલ્મ ફે...,0.908595,આત્મકથામાં ઉલ્લેખ કરવામાં આવ્યોઃ-,0.855426
3,સૌરાષ્ટ્રમાં કિસાનોની કરુણા: પરસેવાથી પકાવેલા ...,શુક્રવારે મોરબીના સામાકાંઠા વિસ્તારમાં ભાગ્યલક...,0.968374,મુખ્યમંત્રી વિજય રૂપાણીએ વિક્રમ સંવતનું નૂતન વ...,0.927727,છોકરો : હવે 10 રૂપિયાની મગફળી માટે શું ભંડારો ...,0.499201
4,"ડ્રેસની બોડીિસ ફીતથી શણગારવામાં આવી હતી, અને સ...","કાલાવાડ રોડ, ૧૫૦ ફૂટ રીંગ રોડ, મવડી થી પાળ ગામ...",0.976449,"કાલાવાડ રોડ, ૧૫૦ ફૂટ રીંગ રોડ, મવડી થી પાળ ગામ...",0.956160,પેરિસ જેક્સન અને કારા ડેલેવેન,0.877544
5,આલિયાએ બીજી વાર રણવીર સિંહ સાથે બનાવી જોડી,વાયરલ થઇ રહેલાં આ વીડિયોમાં આપે ઇન્સ્ટાગ્રામ અ...,0.952996,વાત કરીએ અમદાવાદનો વિસ્મય શાહ હિટ એન્ડ રન કેસ ...,0.899799,પત્રકાર શેખર ગુપ્તાએ હાલમાં જ એક પોસ્ટ શૅર કરી...,0.892419
6,સ્મીમેર હોસ્પિટલમાં ટુંકાગાળામાં 400 થી વધુ બે...,કોરોનાનો સારવારનો કોન્ટ્રાકટ પુરો થતા રાજકોટ શ...,0.938526,લાંબા સમય સુધી ફૂગ-પ્રતિરોધક ગુણધર્મોને રંગવા ...,0.905234,કેટલાક દેશો જે કંગાળ હતા ત્યાંની હાલત વધુ દયની...,0.919472
7,પંગાઃ જ્યારે મીનૂ અરેન્જ મેરેજ માટે એક છોકરા અ...,બબીતાએ તેની ફિલ્મી કરિયરમાં ખૂબ ઓછી ફિલ્મો કરી...,0.983918,હું કબૂલ કરું છું કે મેં છેતરતી અને વારંવાર સા...,0.964387,"ધનવાન બનવાની ઈચ્છા સૌની હોય છે, તમારી પણ હશે. ...",0.626806
8,* તમામ ભારતીયોને પીવાનું શુદ્ધ પાણી મળે તે ...,દૂધ સંઘના મેનેજીંગ ડીરેકટર વિનોદ વ્યાસે સંઘના ...,0.957577,નવું ભારત તેનું આગવું સ્થાન બનાવી રહ્યું છે ત્...,0.910277,મધ્યપ્રદેશના જબલપુરમાંથી મળ્યો રેપર.,0.908733
9,1. વાળની લેન્થ ઓછી થઈ જાય ત્યારે શું કરવું? :,"1.ગરમ પાણીથી સ્નાન કર્યા પછી ત્વચામાં લાલાશ, ફ...",0.922448,બચતખાતામાં મળતું વ્યાજ વ્યક્તિની આવકમાં ગણાય છ...,0.854145,"4. બ્લ્સ પ્રેશર, ડાયાબિટીસનો ઈલાજ કરવો :",0.759081


In [39]:
results_df.to_csv(
    "/content/sentence_similarity_results.csv",
    index=False
)

print("Final sentence similarity results saved!")

Final sentence similarity results saved!
